In [3]:
# precompute_curves.py
import json, numpy as np, torch
from PropertyPredictor import PropertyPredictor

# ---- robust dtype defaults
torch.set_default_dtype(torch.float32)

# ---- load checkpoint
ckpt = torch.load("model_checkpoint/best.pth", weights_only=True)
x_cols = ckpt["x_cols"]
y_cols = ckpt["y_cols"]

# build model and move to float32
model = PropertyPredictor(input_dim=len(x_cols), hidden_dim=1024, output_dim=len(y_cols))
model.load_state_dict(ckpt["state_dict"])
model.eval().float()  # ensure weights are float32

# optional: if these are numpy arrays, coerce to float32 tensors
y_mean = torch.as_tensor(ckpt["y_mean"], dtype=torch.float32)
y_std  = torch.as_tensor(ckpt["y_std"],  dtype=torch.float32)

# ---- load inputs used to define ranges/means
# Use the same dataframe/CSV you used to train/normalize.
# If you have a CSV, load and pick the exact x_cols:
import pandas as pd
DF = pd.read_csv("webapp/data/sample.csv")  # adjust path
X_np = DF[x_cols].to_numpy(dtype=np.float32)  # enforce float32 at the source
X = torch.from_numpy(X_np)                     # shape [N, Din], float32

# ---- compute per-dimension curves
num_points = 100
curves = {}

with torch.no_grad():
    x_ref = X.mean(dim=0)  # [Din], float32

    for i, x_name in enumerate(x_cols):
        # build a sweep along dimension i
        x_test = x_ref.repeat(num_points, 1)               # [num_points, Din]
        xmin, xmax = X[:, i].min().item(), X[:, i].max().item()
        x_test[:, i] = torch.linspace(xmin, xmax, num_points, dtype=torch.float32)

        y_pred = model(x_test)                              # [num_points, Dout], float32
        # de-normalize (if your model outputs were normalized)
        y_pred = y_pred * y_std + y_mean                    # broadcast

        x_curve = x_test[:, i].cpu().numpy()                # (num_points,)
        y_pred_np = y_pred.cpu().numpy()                    # (num_points, Dout)

        # store per-output curves as [[x,y], ...]
        curves[x_name] = {
            y_col: np.c_[x_curve, y_pred_np[:, j]].tolist()
            for j, y_col in enumerate(y_cols)
        }

# ---- write JSON used by the web app
with open("webapp/data/curves.json", "w") as f:
    json.dump(curves, f)
print("Wrote data/curves.json")

Wrote data/curves.json
